# Alpaca-API-kall med Python

In [1]:
# Importerer biblioteker
from dotenv import load_dotenv
import os
import pandas as pd

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from alpaca.trading.client import TradingClient

In [2]:
# Henter nøkkler fra env fil.
load_dotenv()
KEY = os.getenv("key")
SECRET = os.getenv("secret")

## Henter markedsdata fra AAPL (Apple)

### Historisk data

In [3]:
client = StockHistoricalDataClient(KEY, SECRET)

request = StockBarsRequest(
    symbol_or_symbols="AAPL",
    timeframe=TimeFrame.Day,
    start="2025-01-01"
)

bars = client.get_stock_bars(request)

# Omgjørelse til pandas
df = bars.df

print(df)

                                     open      high       low   close  \
symbol timestamp                                                        
AAPL   2025-01-02 05:00:00+00:00  248.930  249.1000  241.8201  243.85   
       2025-01-03 05:00:00+00:00  243.360  244.1800  241.8900  243.36   
       2025-01-06 05:00:00+00:00  244.310  247.3300  243.2000  245.00   
       2025-01-07 05:00:00+00:00  242.980  245.5500  241.3500  242.21   
       2025-01-08 05:00:00+00:00  241.920  243.7123  240.0500  242.70   
...                                   ...       ...       ...     ...   
       2026-06-09 04:00:00+00:00  300.275  300.7500  287.7800  290.55   
       2026-06-10 04:00:00+00:00  290.740  294.7500  287.3800  291.58   
       2026-06-11 04:00:00+00:00  293.720  297.0000  289.5900  295.63   
       2026-06-12 04:00:00+00:00  296.030  297.1400  289.6200  291.13   
       2026-06-15 04:00:00+00:00  294.120  297.7800  291.7000  296.42   

                                      volume  trad

In [4]:
print(df.head(10))

                                     open      high       low   close  \
symbol timestamp                                                        
AAPL   2025-01-02 05:00:00+00:00  248.930  249.1000  241.8201  243.85   
       2025-01-03 05:00:00+00:00  243.360  244.1800  241.8900  243.36   
       2025-01-06 05:00:00+00:00  244.310  247.3300  243.2000  245.00   
       2025-01-07 05:00:00+00:00  242.980  245.5500  241.3500  242.21   
       2025-01-08 05:00:00+00:00  241.920  243.7123  240.0500  242.70   
       2025-01-10 05:00:00+00:00  240.010  240.1600  233.0000  236.85   
       2025-01-13 05:00:00+00:00  233.530  234.6700  229.7200  234.40   
       2025-01-14 05:00:00+00:00  234.750  236.1200  232.4720  233.28   
       2025-01-15 05:00:00+00:00  234.635  238.9600  234.4300  237.87   
       2025-01-16 05:00:00+00:00  237.350  238.0100  228.0300  228.26   

                                      volume  trade_count        vwap  
symbol timestamp                                   

In [5]:
print(df.tail(3))

                                    open    high     low   close      volume  \
symbol timestamp                                                               
AAPL   2026-06-11 04:00:00+00:00  293.72  297.00  289.59  295.63  42760915.0   
       2026-06-12 04:00:00+00:00  296.03  297.14  289.62  291.13  38970821.0   
       2026-06-15 04:00:00+00:00  294.12  297.78  291.70  296.42  45931352.0   

                                  trade_count        vwap  
symbol timestamp                                           
AAPL   2026-06-11 04:00:00+00:00     833140.0  294.111279  
       2026-06-12 04:00:00+00:00     830866.0  291.579339  
       2026-06-15 04:00:00+00:00     792074.0  296.015019  


In [6]:
print(df["close"] / df["open"]) # Kan bruke beregninger

symbol  timestamp                
AAPL    2025-01-02 05:00:00+00:00    0.979593
        2025-01-03 05:00:00+00:00    1.000000
        2025-01-06 05:00:00+00:00    1.002824
        2025-01-07 05:00:00+00:00    0.996831
        2025-01-08 05:00:00+00:00    1.003224
                                       ...   
        2026-06-09 04:00:00+00:00    0.967613
        2026-06-10 04:00:00+00:00    1.002889
        2026-06-11 04:00:00+00:00    1.006503
        2026-06-12 04:00:00+00:00    0.983448
        2026-06-15 04:00:00+00:00    1.007820
Length: 363, dtype: float64


In [7]:
print(df["close"].mean())   # Gjennomsnittet
print(df["close"].std())    # Standardaviket

244.52358126721762
30.464475659271475


## Henter Account Information

In [8]:
trading_client = TradingClient(
    KEY, SECRET
)

account = trading_client.get_account()

print(account.equity)
print(account.buying_power)

1008158.12
751063.91


In [9]:
positions = trading_client.get_all_positions()

total_unrealized_pl = 0
total_cost_basis = 0
for i in positions:
    print(
        i.symbol,
        i.unrealized_pl,
        i.qty
    )
    total_unrealized_pl += float(i.unrealized_pl)
    total_cost_basis += float(i.cost_basis)

print(account.equity)
print(float(account.cash) + total_unrealized_pl + total_cost_basis)

AAPL -13.03 1
AVGO -12830.550496 1312
GOOGL 359.220003 23
PLTR -90898.644705 9735
TSLA -462.430005 17
1008158.12
1008158.1199999999


In [10]:
# Hente portfølje history
trading_client.get_portfolio_history()

{   'base_value': 1109562.61,
    'cashflow': {},
    'equity': [   1077131.03,
                  1082189.32,
                  1064812.85,
                  1106188.9,
                  1104549.6,
                  1098896.14,
                  1118061.64,
                  1075840.79,
                  1195019.97,
                  1334840.05,
                  1384238.98,
                  1303431.5,
                  1208388.36,
                  1141846.64,
                  979019.8,
                  1012482.38,
                  972606.94,
                  927744.38,
                  954221.01,
                  919711.05,
                  1001011.63],
    'profit_loss': [   -32431.58,
                       5058.29,
                       -17376.47,
                       41376.05,
                       -1639.3,
                       -5653.46,
                       19165.5,
                       -42220.85,
                       119179.18,
                       139820.


### Cash

Kontanter tilgjengelig på kontoen som ikke er investert i åpne posisjoner.

<br>

### Cost Basis

Det opprinnelige beløpet som ble investert i en posisjon:

$$
Cost\ Basis =
Entry\ Price \times Quantity
$$

<br>

### Market Value

Den nåværende markedsverdien av en åpen posisjon:

$$
Market\ Value =
Current\ Price \times Quantity
$$

<br>

### Unrealized PnL

Gevinst eller tap på åpne posisjoner som ennå ikke er solgt:

$$
Unrealized\ PnL =
Market\ Value - Cost\ Basis
$$

eller

$$
Unrealized\ PnL =
(Current\ Price - Entry\ Price)
\times Quantity
$$

<br>

### Equity

Verdien av kontoen inkludert kontanter og markedsverdien av alle åpne posisjoner:

$$
Equity =
Cash + Market\ Value
$$

eller ekvivalent:

$$
Equity =
Cash + Cost\ Basis + Unrealized\ PnL
$$

### Daily Change

Prosentvis endring i kontoverdi siden forrige handelsdag:

$$
Daily\ Change=
\frac{Equity_{today}-Equity_{yesterday}}
{Equity_{yesterday}}
\times100
$$


<br>

### Realized PnL

Gevinst eller tap som er låst inn ved at en posisjon er avsluttet:
$$
Realized\ PnL =
(Sell\ Price - Buy\ Price)
\times Quantity
$$

<br>

### Initial Margin

Minimumskravet til egenkapital som kreves for å åpne en ny posisjon:

$$
Initial\ Margin =
Position\ Value \times Initial\ Margin\ Requirement
$$

<br>

### Maintenance Margin

Minimum egenkapital som må opprettholdes for å beholde en åpen posisjon:
$$
Maintenance\ Margin =
Position\ Value \times Maintenance\ Margin\ Requirement
$$

<br>


### Margin Cushion

Hvor mye egenkapital som er tilgjengelig utover maintenance margin-kravet:

$$
Margin\ Cushion =
\frac{Equity - Maintenance\ Margin}
{Equity}
$$

<br>

### Account Leverage

Forholdet mellom eksponering i markedet og egenkapital:

$$
Account\ Leverage =
\frac{Market\ Value}{Equity}
$$

<br>

### Buying Power

Hvor stor markedsverdi som kan kjøpes med tilgjengelig kapital og margin:

$$
Buying\ Power =
Equity \times Leverage
$$